# 03 — Train on Colab (GPU)

Runs the **real** training for all three models on a free Colab GPU and
prints a results table to paste back into the README + model card.

**How to use:**
1. Open https://colab.research.google.com → File → Upload notebook → this file.
2. Runtime → Change runtime type → **T4 GPU** → Save.
3. Runtime → **Run all**.
4. Total time ≈ 15 minutes. The last cell prints a Markdown table — copy it.

The notebook clones the public repo and trains using the project's own
`sarcasm_radar` modules, so the numbers come from the real codebase, not
from notebook-only code.

## 1. Setup — clone the repo and install

In [ ]:
import inspect
import os
import shutil
import subprocess
import sys

# Don't stand inside the directory we are about to delete.
os.chdir("/content")

# 1. Remove ANY previous clone (including a nested sarcasm-radar/sarcasm-radar
#    left over from an earlier re-run).
shutil.rmtree("/content/sarcasm-radar", ignore_errors=True)

# 2. Fresh clone to an explicit absolute path.
subprocess.run(
    [
        "git", "clone", "-q",
        "https://github.com/Jenisssh/sarcasm-radar.git",
        "/content/sarcasm-radar",
    ],
    check=True,
)

# 3. Drop any stale sarcasm_radar modules cached from an earlier run, so the
#    next import re-reads from the fresh clone.
for _name in [m for m in list(sys.modules) if m == "sarcasm_radar" or m.startswith("sarcasm_radar.")]:
    del sys.modules[_name]

# 4. Point sys.path only at the fresh clone.
sys.path = [p for p in sys.path if "sarcasm-radar" not in p]
sys.path.insert(0, "/content/sarcasm-radar/src")

# 5. Install ML deps (Colab already ships torch, pandas, numpy, scikit-learn).
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.46,<4.48", "datasets>=3.1,<3.2", "accelerate>=1.1,<1.3",
    ],
    check=True,
)

# 6. Import and HARD-VERIFY the processing_class fix is the code that loaded.
#    If this assert trips, Colab is running stale code -> delete the runtime.
import sarcasm_radar
from sarcasm_radar.models import transformer

_fit_src = inspect.getsource(transformer.TransformerSarcasmClassifier.fit)
assert "processing_class" in _fit_src, (
    "STALE CODE: Colab loaded an old transformer.py. "
    "Runtime -> Disconnect and delete runtime, then Run all again."
)

os.chdir("/content/sarcasm-radar")
print(f"sarcasm_radar {sarcasm_radar.__version__} - processing_class fix verified, ready")

In [ ]:
import torch

device = "GPU: " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
print(device)
if not torch.cuda.is_available():
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")

## 2. Load the corpus

Primary source is the **SemEval-2022 Task 6 iSarcasmEval** English tweet
set — it matches the project's tweet framing. If that URL doesn't
resolve, the notebook falls back to a HuggingFace news-headlines
sarcasm dataset and prints which source it used.

The 50-row curated Indian English supplement is loaded separately and
held out entirely as a **cross-domain probe** — the model never trains
on it, so `en-IN / hi-en` macro-F1 is an honest generalisation test.

In [ ]:
import pandas as pd


def load_main_corpus() -> tuple[pd.DataFrame, str]:
    """Return (corpus_df with text+label, source_name). Tries SemEval, falls back to headlines."""
    for branch in ("main", "master"):
        url = f"https://raw.githubusercontent.com/iabufarha/iSarcasmEval/{branch}/train/train.En.csv"
        try:
            df = pd.read_csv(url)
            if {"tweet", "sarcastic"}.issubset(df.columns):
                df = df.dropna(subset=["tweet"])[["tweet", "sarcastic"]]
                df = df.rename(columns={"tweet": "text", "sarcastic": "label"})
                return df, "SemEval-2022 iSarcasmEval (English tweets)"
        except Exception as e:  # noqa: BLE001
            print(f"  iSarcasmEval/{branch} unavailable: {e}")
    # Fallback
    from datasets import load_dataset

    ds = load_dataset("raquiba/Sarcasm_News_Headline", split="train")
    df = ds.to_pandas()
    text_col = "headline" if "headline" in df.columns else "text"
    df = df.rename(columns={text_col: "text", "is_sarcastic": "label"})
    return df[["text", "label"]], "News Headlines sarcasm (HuggingFace)"


corpus, source_name = load_main_corpus()
corpus["text"] = corpus["text"].astype(str)
corpus["label"] = corpus["label"].astype(int)
print(f"\nSOURCE USED: {source_name}")
print(f"rows: {len(corpus):,}   class balance: {corpus['label'].mean():.3f} positive")
corpus.head()

In [ ]:
from sklearn.model_selection import train_test_split

from sarcasm_radar.data.curate import load_curated_labels

train_df, test_df = train_test_split(
    corpus, test_size=0.2, stratify=corpus["label"], random_state=42
)

# Curated Indian English supplement — held-out cross-domain probe.
probe = load_curated_labels()

print(f"train: {len(train_df):,}")
print(f"test : {len(test_df):,}")
print(f"probe (en-IN / hi-en): {len(probe)}  — model never trains on these")

## 3. Baseline — TF-IDF + Logistic Regression

In [ ]:
from sarcasm_radar.evaluation.metrics import evaluate
from sarcasm_radar.models.baseline import train_baseline

baseline = train_baseline(train_df["text"], train_df["label"])

base_test = evaluate(test_df["label"], baseline.predict(test_df["text"]))
base_probe = evaluate(probe["label"], baseline.predict(probe["text"]))
print(f"baseline  test macro-F1 : {base_test.macro_f1:.4f}")
print(f"baseline  probe macro-F1: {base_probe.macro_f1:.4f}")

## 4. DistilBERT fine-tune

~3-5 minutes on a T4.

In [ ]:
from sarcasm_radar.models.transformer import (
    TransformerSarcasmClassifier,
    TransformerTrainConfig,
)

distilbert = TransformerSarcasmClassifier(
    TransformerTrainConfig(num_epochs=3, batch_size=16)
)
distilbert.fit(
    train_df["text"], train_df["label"], test_df["text"], test_df["label"]
)

distil_test = evaluate(test_df["label"], distilbert.predict(test_df["text"]))
distil_probe = evaluate(probe["label"], distilbert.predict(probe["text"]))
print(f"distilbert  test macro-F1 : {distil_test.macro_f1:.4f}")
print(f"distilbert  probe macro-F1: {distil_probe.macro_f1:.4f}")

## 5. XLM-RoBERTa fine-tune

~8-12 minutes on a T4 — bigger model, one extra epoch.

In [ ]:
from sarcasm_radar.models.multilingual import make_xlmr_config

xlmr = TransformerSarcasmClassifier(make_xlmr_config())
xlmr.fit(train_df["text"], train_df["label"], test_df["text"], test_df["label"])

xlmr_test = evaluate(test_df["label"], xlmr.predict(test_df["text"]))
xlmr_probe = evaluate(probe["label"], xlmr.predict(probe["text"]))
print(f"xlm-roberta  test macro-F1 : {xlmr_test.macro_f1:.4f}")
print(f"xlm-roberta  probe macro-F1: {xlmr_probe.macro_f1:.4f}")

## 6. Per-register breakdown on the probe set

In [ ]:
from sarcasm_radar.models.multilingual import compare_per_register

per_register = compare_per_register(
    probe["label"],
    {
        "baseline": baseline.predict(probe["text"]),
        "distilbert": distilbert.predict(probe["text"]),
        "xlm-roberta": xlmr.predict(probe["text"]),
    },
    registers=probe["register"],
)
per_register.round(4)

## 7. Results — copy everything below this cell's output

In [ ]:
print("=" * 60)
print(f"DATA SOURCE: {source_name}")
print(f"train rows: {len(train_df):,}   test rows: {len(test_df):,}   probe rows: {len(probe)}")
print("=" * 60)
print()
print("| Model | Macro-F1 (test) | Macro-F1 (en-IN probe) |")
print("|-------|:-:|:-:|")
print(f"| TF-IDF + LR | {base_test.macro_f1:.3f} | {base_probe.macro_f1:.3f} |")
print(f"| DistilBERT | {distil_test.macro_f1:.3f} | {distil_probe.macro_f1:.3f} |")
print(f"| XLM-RoBERTa | {xlmr_test.macro_f1:.3f} | {xlmr_probe.macro_f1:.3f} |")
print()
print("Per-class F1 (XLM-RoBERTa, test set):")
print(f"  not-sarcastic: {xlmr_test.not_sarcastic.f1:.3f}")
print(f"  sarcastic    : {xlmr_test.sarcastic.f1:.3f}")